# LoRA Implementation Exercises

This notebook implements Low-Rank Adaptation (LoRA) from scratch in PyTorch, integrates it into linear layers and a small MLP, then freezes the original weights so only LoRA parameters are fine-tuned.

## 0. Setup

In [ ]:
!pip install -q torch torchvision matplotlib

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

random_seed = 123
torch.manual_seed(random_seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Exercise 1: Implementing the LoRALayer

LoRA freezes the original model weights and learns a low-rank update. Instead of training a full weight matrix of shape `(in_dim, out_dim)`, LoRA learns two smaller matrices:

- `A`: `(in_dim, rank)`
- `B`: `(rank, out_dim)`

The adaptation is computed as `x @ A @ B`, scaled by `alpha / rank`. Matrix `B` is initialized to zeros so the LoRA branch initially produces no change to the original model.

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        if rank <= 0:
            raise ValueError("rank must be greater than 0")

        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

    def forward(self, x):
        return self.scaling * (x @ self.A @ self.B)


lora = LoRALayer(in_dim=4, out_dim=3, rank=2, alpha=4)
x_small = torch.randn(5, 4)
y_small = lora(x_small)

print("Input shape :", x_small.shape)
print("Output shape:", y_small.shape)
print("Initial LoRA output should be all zeros because B is initialized to zero:")
print(y_small)

## Exercise 2: Implementing the LinearWithLoRA Layer

`LinearWithLoRA` wraps an existing `nn.Linear` layer and adds the low-rank adaptation to the normal linear output.

In [ ]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


torch.manual_seed(random_seed)
layer = nn.Linear(4, 3)
x = torch.randn(2, 4)

print("Input:")
print(x)
print("\nOriginal layer:")
print(layer)
print("\nOriginal output:")
print(layer(x))

layer_lora_1 = LinearWithLoRA(copy.deepcopy(layer), rank=2, alpha=4)
print("\nLinearWithLoRA output:")
print(layer_lora_1(x))

max_difference = (layer(x) - layer_lora_1(x)).abs().max().item()
print("\nMax difference at initialization:", max_difference)

## Exercise 3: Creating a Small Neural Network and Applying LoRA

Because the LoRA `B` matrix starts at zero, replacing a linear layer with `LinearWithLoRA` should initially preserve the original output exactly.

In [ ]:
class SingleLayerNet(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.fc(x)


torch.manual_seed(random_seed)
base_net = SingleLayerNet(4, 3)
lora_net = copy.deepcopy(base_net)
lora_net.fc = LinearWithLoRA(lora_net.fc, rank=2, alpha=4)

test_input = torch.randn(6, 4)
base_output = base_net(test_input)
lora_output = lora_net(test_input)

print("Base output:")
print(base_output)
print("\nLoRA output:")
print(lora_output)
print("\nOutputs unchanged initially:", torch.allclose(base_output, lora_output, atol=1e-6))

## Exercise 4: Merging LoRA Matrices and Testing Equivalence

For efficient inference, LoRA can be merged into the original weight matrix. Since `nn.Linear` stores weights as `(out_features, in_features)`, the LoRA update `(in_features, out_features)` must be transposed before being added.

In [ ]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def merged_weight(self):
        lora_update = self.lora.A @ self.lora.B
        return self.linear.weight + self.lora.scaling * lora_update.T

    def forward(self, x):
        return F.linear(x, self.merged_weight(), self.linear.bias)


layer_lora_2 = LinearWithLoRAMerged(copy.deepcopy(layer), rank=2, alpha=4)
layer_lora_2.linear.load_state_dict(layer_lora_1.linear.state_dict())
layer_lora_2.lora.load_state_dict(layer_lora_1.lora.state_dict())

output_unmerged = layer_lora_1(x)
output_merged = layer_lora_2(x)

print("Unmerged LoRA output:")
print(output_unmerged)
print("\nMerged LoRA output:")
print(output_merged)
print("\nMerged and unmerged are equivalent:", torch.allclose(output_unmerged, output_merged, atol=1e-6))

## Exercise 5: Implementing an MLP and Replacing Layers with LoRA

We build a 3-layer MLP for MNIST. The input image is flattened from `1 x 28 x 28` to `784` features.

In [ ]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.layers(x)


def replace_linear_with_lora(module, rank=4, alpha=8):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            setattr(module, name, LinearWithLoRAMerged(child, rank=rank, alpha=alpha))
        else:
            replace_linear_with_lora(child, rank=rank, alpha=alpha)


# Architecture
num_features = 28 * 28
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

# Settings
learning_rate = 1e-3
num_epochs = 2

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
).to(DEVICE)

model_lora_architecture = copy.deepcopy(model)
replace_linear_with_lora(model_lora_architecture, rank=4, alpha=8)

print("Original MLP:")
print(model)
print("\nMLP with LoRA layers:")
print(model_lora_architecture)

## Loading Dataset

The exercise uses MNIST. To keep the notebook quick, the DataLoaders below train on a subset. Remove the `Subset` lines to train on the full dataset.

In [ ]:
BATCH_SIZE = 64

transform = transforms.ToTensor()
train_dataset_full = datasets.MNIST(root="data", train=True, transform=transform, download=True)
test_dataset_full = datasets.MNIST(root="data", train=False, transform=transform, download=True)

train_dataset = Subset(train_dataset_full, range(5000))
test_dataset = Subset(test_dataset_full, range(1000))

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print("Image batch dimensions:", images.shape)
    print("Image label dimensions:", labels.shape)
    break

plt.figure(figsize=(8, 2))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(str(labels[i].item()))
    plt.axis("off")
plt.show()

## Define Evaluation and Training

In [ ]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0

    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            _, predicted_labels = torch.max(logits, dim=1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()

    return correct_pred / num_examples * 100


def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()

        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_idx % 40 == 0:
                print(
                    "Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f"
                    % (epoch + 1, num_epochs, batch_idx, len(train_loader), loss.item())
                )

        train_acc = compute_accuracy(model, train_loader, device)
        print("Epoch: %03d/%03d training accuracy: %.2f%%" % (epoch + 1, num_epochs, train_acc))
        print("Time elapsed: %.2f min" % ((time.time() - start_time) / 60))

    print("Total Training Time: %.2f min" % ((time.time() - start_time) / 60))

## Train the Base MLP

First we train a normal MLP. Then we copy it and add LoRA layers to simulate efficient adaptation.

In [ ]:
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(model)
print(optimizer_pretrained)

train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f"Test accuracy base model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")

## Replacing Linear Layers with LoRA Layers

We copy the trained base model and replace every `nn.Linear` layer with `LinearWithLoRAMerged`. Because `B` is initialized to zero, the LoRA version should initially behave like the base model.

In [ ]:
model_lora = copy.deepcopy(model)
replace_linear_with_lora(model_lora, rank=4, alpha=8)
model_lora.to(DEVICE)

print(model_lora)
print(f"Test accuracy original model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model before fine-tuning: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

## Exercise 6: Freezing the Original Linear Layers and Training LoRA

The original `nn.Linear` parameters inside each LoRA wrapper are frozen. Only the low-rank matrices `A` and `B` remain trainable.

In [ ]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)


def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
        print(f"{name:45s} trainable={param.requires_grad}")
    print(f"\nTrainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


freeze_linear_layers(model_lora)
print_trainable_parameters(model_lora)

In [ ]:
optimizer_lora = torch.optim.Adam(
    [param for param in model_lora.parameters() if param.requires_grad],
    lr=learning_rate
)

train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)

print(f"Test accuracy original model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA fine-tuned model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

## Final Reflection

LoRA reduces the number of trainable parameters by learning a low-rank update instead of modifying the full original weight matrices. In this notebook, the original MLP can be trained normally, then LoRA adapters can be added while preserving the initial outputs because the `B` matrix starts at zero. After freezing the original linear layers, only the LoRA matrices are updated during fine-tuning.

This workflow is especially useful for large models. Training every parameter in a large Transformer can be expensive, but LoRA lets us adapt the model with a much smaller number of parameters. Merging the LoRA update into the original weight matrix also makes inference efficient because the adapted model can run like a standard linear layer.